# Train BERT on Google Colab

This notebook runs in Google Colab to fine-tune a BERT model for comment sentiment analysis. It mounts Google Drive, installs required packages, loads or accepts a dataset, runs `backend/train_bert.py` (from the repo or uploaded files), and saves the fine-tuned model to Drive.

Instructions:
- If you have a GitHub repo URL, provide it in the "Clone repo" cell.
- Or upload `train_bert.py` and your dataset `cleaned_comments.csv` into the notebook environment.
- Use a GPU runtime (Runtime -> Change runtime type -> GPU) for faster training.


In [ ]:
# Cell 2: Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Path where Drive will be mounted
DRIVE_ROOT = '/content/drive/MyDrive'
print('Drive mounted at', DRIVE_ROOT)

## Install dependencies
Run the following to install required Python packages in Colab. Use `pip` in Colab to install packages needed by `train_bert.py`.

In [ ]:
!pip install --upgrade pip
!pip install transformers==4.40.0 datasets torch scikit-learn nltk pandas firebase-admin google-api-python-client google-auth-oauthlib tqdm pillow

## Clone repository (optional)
Provide your GitHub repo URL or skip this cell if you will upload `train_bert.py` and the dataset manually.

In [ ]:
#@markdown Enter GitHub repo URL (or leave empty to upload files manually)
GITHUB_REPO = ""  # @param {type:"string"}

if GITHUB_REPO:
    !git clone {GITHUB_REPO} repo
    %cd repo
else:
    print('No repo provided — please upload train_bert.py and cleaned_comments.csv')


## Upload files (if not cloning)
Use the file upload widget to upload `train_bert.py` and `cleaned_comments.csv` into the Colab environment.

In [ ]:
from google.colab import files

# Upload files if user didn't clone the repo
uploaded = files.upload()
for fn in uploaded:
    print('Uploaded:', fn)


## Prepare dataset and directories
Ensure the dataset is available at `./backend/data/cleaned_comments.csv`. Create folders for models and logs.

In [ ]:
import os
os.makedirs('backend/data', exist_ok=True)
os.makedirs('backend/models', exist_ok=True)

# If user uploaded cleaned_comments.csv, move it into backend/data
if 'cleaned_comments.csv' in uploaded:
    os.rename('cleaned_comments.csv', 'backend/data/cleaned_comments.csv')

print('Data folder contents:', os.listdir('backend/data'))

## Run training
This cell runs the repository training script `backend/train_bert.py`. Adjust `TRAINING_ARGS` if you want to pass different hyperparameters.

In [ ]:
#@markdown Adjust training options below
TRAINING_ARGS = "--epochs 1 --batch_size 16"  # @param {type:"string"}

# Run the training script from the repo or uploaded files
if os.path.exists('backend/train_bert.py'):
    !python backend/train_bert.py $TRAINING_ARGS
else:
    print('backend/train_bert.py not found — ensure you cloned the repo or uploaded the file')

## Save model to Google Drive
After training completes, this cell copies the model folder to your Drive.

In [ ]:
import shutil
DRIVE_MODELS_DIR = DRIVE_ROOT + '/finetuned_models'
os.makedirs(DRIVE_MODELS_DIR, exist_ok=True)

if os.path.exists('backend/models/finetuned_bert'):
    dest = DRIVE_MODELS_DIR + '/finetuned_bert'
    if os.path.exists(dest):
        shutil.rmtree(dest)
    shutil.copytree('backend/models/finetuned_bert', dest)
    print('Saved model to', dest)
else:
    print('No finetuned model found at backend/models/finetuned_bert')